# Lakelink Fiber — Provable Resilience Intelligence
## Executive Summary Analysis

**Portfolio:** 8,000 enterprise services | **Revenue at Risk:** $62.88M/mo | **Avg Resilience Score:** 50.0/100

This notebook provides the executive-level analysis powering the Provable Resilience Intelligence demo. All data sourced from `cdm_tmforum.provable_resilience_intelligence`.

In [0]:
%sql
-- Executive KPI Summary
SELECT 
    COUNT(*) AS total_services,
    ROUND(SUM(revenue_at_risk_usd)/1e6, 2) AS revenue_at_risk_M,
    ROUND(AVG(overall_score), 1) AS avg_resilience_score,
    SUM(CASE WHEN overall_score < 40 THEN 1 ELSE 0 END) AS critical_services,
    SUM(CASE WHEN overall_score >= 40 AND overall_score < 60 THEN 1 ELSE 0 END) AS high_risk,
    SUM(CASE WHEN overall_score >= 60 THEN 1 ELSE 0 END) AS medium_or_better,
    ROUND(MIN(overall_score), 1) AS min_score,
    ROUND(MAX(overall_score), 1) AS max_score
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot

In [0]:
%sql
-- Revenue at Risk by Risk Band
SELECT 
    CASE 
        WHEN overall_score < 40 THEN 'Critical (<40)'
        WHEN overall_score < 60 THEN 'High (40-59)'
        WHEN overall_score < 80 THEN 'Medium (60-79)'
        ELSE 'Low (80+)'
    END AS risk_band,
    COUNT(*) AS services,
    ROUND(SUM(revenue_at_risk_usd)/1e6, 2) AS revenue_at_risk_M,
    ROUND(AVG(overall_score), 1) AS avg_score,
    ROUND(AVG(telemetry_health_score), 1) AS avg_telemetry,
    ROUND(AVG(path_diversity_score), 1) AS avg_diversity,
    ROUND(AVG(hazard_exposure_score), 1) AS avg_hazard
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot
GROUP BY 1
ORDER BY CASE 
    WHEN overall_score < 40 THEN 1 
    WHEN overall_score < 60 THEN 2 
    WHEN overall_score < 80 THEN 3 
    ELSE 4 END

In [0]:
%sql
-- Top 10 Riskiest Services by Revenue at Risk
SELECT 
    s.service_id,
    ROUND(s.overall_score, 1) AS score,
    ROUND(s.path_diversity_score, 1) AS diversity,
    ROUND(s.hazard_exposure_score, 1) AS hazard,
    ROUND(s.telemetry_health_score, 1) AS telemetry,
    ROUND(s.revenue_at_risk_usd, 0) AS revenue_at_risk,
    si.service_type,
    si.sla_tier,
    si.sold_as_diverse,
    c.customer_name,
    c.industry,
    c.region
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot s
JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory si ON s.service_id = si.service_id
JOIN cdm_tmforum.provable_resilience_intelligence.customer c ON si.customer_id = c.customer_id
ORDER BY s.revenue_at_risk_usd DESC
LIMIT 10

In [0]:
%sql
-- Top 10 Industry/Region Corridors by Revenue at Risk
SELECT 
    c.industry,
    c.region,
    CONCAT(c.industry, ' / ', c.region) AS corridor,
    COUNT(*) AS services,
    ROUND(SUM(s.revenue_at_risk_usd)/1e6, 2) AS revenue_at_risk_M,
    ROUND(AVG(s.overall_score), 1) AS avg_score,
    SUM(CASE WHEN s.overall_score < 40 THEN 1 ELSE 0 END) AS critical_count,
    SUM(CASE WHEN si.sold_as_diverse THEN 1 ELSE 0 END) AS diverse_count
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot s
JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory si ON s.service_id = si.service_id
JOIN cdm_tmforum.provable_resilience_intelligence.customer c ON si.customer_id = c.customer_id
GROUP BY c.industry, c.region
ORDER BY SUM(s.revenue_at_risk_usd) DESC
LIMIT 10

In [0]:
%sql
-- SLA Breach Risk: False Diversity by SLA Tier
-- Services sold with diversity guarantees but sharing physical infrastructure (SRLGs)
SELECT 
    si.sla_tier,
    COUNT(*) AS total_services,
    SUM(CASE WHEN si.sold_as_diverse THEN 1 ELSE 0 END) AS sold_diverse,
    SUM(CASE WHEN si.sold_as_diverse AND s.path_diversity_score < 100 THEN 1 ELSE 0 END) AS false_diversity,
    ROUND(100.0 * SUM(CASE WHEN si.sold_as_diverse AND s.path_diversity_score < 100 THEN 1 ELSE 0 END) / 
          NULLIF(SUM(CASE WHEN si.sold_as_diverse THEN 1 ELSE 0 END), 0), 1) AS breach_risk_pct,
    ROUND(SUM(CASE WHEN si.sold_as_diverse AND s.path_diversity_score < 100 THEN s.revenue_at_risk_usd ELSE 0 END)/1e6, 2) AS false_div_rar_M
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot s
JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory si ON s.service_id = si.service_id
GROUP BY si.sla_tier
ORDER BY si.sla_tier

In [0]:
%sql
-- False Diversity Impact: Services sold as diverse but sharing SRLGs
SELECT 
    CASE WHEN si.sold_as_diverse AND s.path_diversity_score < 100 THEN 'False Diversity'
         WHEN si.sold_as_diverse THEN 'True Diversity'
         ELSE 'Not Sold Diverse'
    END AS diversity_status,
    COUNT(*) AS services,
    ROUND(SUM(s.revenue_at_risk_usd)/1e6, 2) AS total_rar_M,
    ROUND(AVG(s.overall_score), 1) AS avg_score,
    ROUND(AVG(s.path_diversity_score), 1) AS avg_diversity_score
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot s
JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory si ON s.service_id = si.service_id
GROUP BY 1
ORDER BY SUM(s.revenue_at_risk_usd) DESC

In [0]:
%sql
-- Top 10 Shared Risk Group Hotspots
SELECT 
    srg.srlg_id,
    srg.srlg_type,
    COUNT(DISTINCT sp.service_id) AS affected_services,
    ROUND(SUM(DISTINCT rs.revenue_at_risk_usd)/1e6, 2) AS revenue_at_risk_M,
    ROUND(AVG(DISTINCT rs.overall_score), 1) AS avg_score
FROM cdm_tmforum.provable_resilience_intelligence.shared_risk_group srg
JOIN cdm_tmforum.provable_resilience_intelligence.service_path sp ON srg.span_id = sp.span_id
JOIN cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot rs ON sp.service_id = rs.service_id
GROUP BY srg.srlg_id, srg.srlg_type
ORDER BY COUNT(DISTINCT sp.service_id) DESC
LIMIT 10

In [0]:
%sql
-- Proactive Action Completion Rates by Type
SELECT 
    action_type,
    COUNT(*) AS total_actions,
    SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) AS completed,
    SUM(CASE WHEN status = 'pending' THEN 1 ELSE 0 END) AS pending,
    SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled,
    ROUND(100.0 * SUM(CASE WHEN status = 'completed' THEN 1 ELSE 0 END) / COUNT(*), 1) AS completion_rate_pct
FROM cdm_tmforum.provable_resilience_intelligence.proactive_action
GROUP BY action_type
ORDER BY COUNT(*) DESC

In [0]:
%sql
-- Revenue at Risk by SLA Tier × Diversity Status
SELECT 
    si.sla_tier,
    si.sold_as_diverse,
    COUNT(*) AS services,
    ROUND(SUM(s.revenue_at_risk_usd)/1e6, 2) AS revenue_at_risk_M,
    ROUND(AVG(s.overall_score), 1) AS avg_score
FROM cdm_tmforum.provable_resilience_intelligence.resilience_score_snapshot s
JOIN cdm_tmforum.provable_resilience_intelligence.service_inventory si ON s.service_id = si.service_id
GROUP BY si.sla_tier, si.sold_as_diverse
ORDER BY si.sla_tier, si.sold_as_diverse